<a href="https://colab.research.google.com/github/peteparker123/graph-rag/blob/main/graph_rag%5B1%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --quiet  json-repair networkx langchain-core langchain-experimental langchain-community

In [ ]:
!pip install "langchain-graph-retriever"

In [ ]:
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.7 MB/s eta 0:00:00


In [ ]:
import json
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)

from langchain_graph_retriever import GraphRetriever
from graph_retriever.strategies import Eager

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.3 MB/s eta 0:00:00


In [ ]:
loader = PyPDFLoader("/content/resume_finale.pdf")

documents = loader.load()

print(f"Loaded {len(documents)} PDF pages")

Loaded 1 PDF pages


In [ ]:
# ============================================================
# 3. SPLIT DOCUMENTS INTO CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 3 chunks


In [ ]:
import json
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)

from langchain_graph_retriever import GraphRetriever
from graph_retriever.strategies import Eager

# ============================================================
# 4. DEFINE METADATA STRUCTURE
# ============================================================

class ChunkMetadata(BaseModel):

    topics: list[str] = Field(
        description="Main topics discussed in the document"
    )

    people: list[str] = Field(
        description="Names of people mentioned in the document"
    )

    organizations: list[str] = Field(
        description="Organizations or companies mentioned"
    )

    technologies: list[str] = Field(
        description="Technologies, tools, frameworks, or programming languages"
    )


# ============================================================
# 5. CREATE GEMINI LLM
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key="your gemini api key",
    temperature=0,
)


# Gemini will return structured ChunkMetadata

metadata_llm = llm.with_structured_output(ChunkMetadata)


# ============================================================
# 6. GENERATE METADATA FOR EACH CHUNK
# ============================================================

print("\nGenerating metadata...\n")


Generating metadata...



In [ ]:
for index, chunk in enumerate(chunks):

    prompt = f"""
Extract useful metadata from the following document.

Rules:

1. Extract only information explicitly present in the document.
2. Do not invent information.
3. Keep metadata values concise.
4. Use consistent naming across documents.

DOCUMENT:

{chunk.page_content}
"""

    metadata = metadata_llm.invoke(prompt)


    # Add generated metadata to existing metadata

    chunk.metadata.update(
        {
            "topics": metadata.topics,
            "people": metadata.people,
            "organizations": metadata.organizations,
            "technologies": metadata.technologies,
        }
    )


    print(f"Processed chunk {index + 1}/{len(chunks)}")

    print("Metadata:")

    print(
        json.dumps(
            chunk.metadata,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("-" * 60)



Processed chunk 1/3
Metadata:
{
  "producer": "Microsoft® Word 2021",
  "creator": "Microsoft® Word 2021",
  "creationdate": "2026-07-03T10:44:14+05:30",
  "author": "Un-named",
  "moddate": "2026-07-03T10:44:14+05:30",
  "source": "/content/resume_finale.pdf",
  "total_pages": 1,
  "page": 0,
  "page_label": "1",
  "topics": [
    "AI systems",
    "Research",
    "Real-world problems",
    "Large Language Models",
    "Deep Learning",
    "Agentic AI",
    "Electronics & Computer Science Engineering",
    "Open-source contribution"
  ],
  "people": [
    "JAI AKASH"
  ],
  "organizations": [
    "Amrita School of Engineering",
    "Hugging Face"
  ],
  "technologies": [
    "transformer-based models",
    "RAG pipelines",
    "Python",
    "C",
    "PyTorch",
    "TensorFlow",
    "Transformers",
    "Scikit-learn",
    "OpenCV",
    "Ultralytics",
    "LangChain",
    "LangGraph",
    "LlamaIndex",
    "FAISS",
    "ChromaDB",
    "Streamlit",
    "Gradio",
    "n8n",
    "WandB",
 

In [ ]:
# ============================================================
# 7. CREATE EMBEDDING MODEL
# ============================================================

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    api_key="your gemini api key"
)


# ============================================================
# 8. CREATE VECTOR STORE
# ============================================================

vector_store = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print("\nDocuments stored in Vector Store")



Documents stored in Vector Store


In [ ]:
# ============================================================
# 9. CREATE GRAPH RETRIEVER
# ============================================================

graph_retriever = GraphRetriever(

    store=vector_store,

    edges=[
        ("topics", "topics"),
        ("people", "people"),
        ("organizations", "organizations"),
        ("technologies", "technologies"),
    ],

    strategy=Eager(
        k=5,
        start_k=1,
        max_depth=2,
    ),
)


In [ ]:
# ============================================================
# 10. ASK USER QUESTION
# ============================================================

question = input("\nEnter your question: ")


# ============================================================
# 11. GRAPH RETRIEVAL
# ============================================================

retrieved_documents = graph_retriever.invoke(question)


# ============================================================
# 12. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print("\n\nRETRIEVED DOCUMENTS")
print("=" * 60)


for index, document in enumerate(retrieved_documents):

    print(f"\nDOCUMENT {index + 1}")

    print("\nCONTENT:")

    print(document.page_content)

    print("\nMETADATA:")

    print(
        json.dumps(
            document.metadata,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("-" * 60)


# ============================================================
# 13. CREATE CONTEXT
# ============================================================

context = "\n\n".join(
    document.page_content
    for document in retrieved_documents
)


# ============================================================
# 14. SEND RETRIEVED CONTEXT TO GEMINI
# ============================================================

answer_prompt = f"""
Answer the user's question using only the provided context.

Carefully examine ALL retrieved documents before answering.

Identify every project relevant to the question, including projects
whose descriptions or technologies imply relevance even if the exact
term from the question is not present in the project title.

For each relevant project:
1. Give the project name.
2. Explain why it is relevant.
3. List the technologies used.

Do not omit relevant projects.

CONTEXT:

{context}

QUESTION:

{question}
"""

response = llm.invoke(answer_prompt)


# ============================================================
# 15. PRINT FINAL ANSWER
# ============================================================

print("\n\nFINAL ANSWER")
print("=" * 60)

print(response.content)


Enter your question: Which projects demonstrate Jai Akash’s experience with RAG and what technologies were used in those projects?


RETRIEVED DOCUMENTS

DOCUMENT 1

CONTENT:
JAI AKASH 
github.com/peteparker123 | huggingface.co/peteparker456 | linkedin.com/in/jai-akash-a3b7a0272 
PROFILE 
Fourth-year Electronics & Computer Science Engineering student passionate about building innovative AI systems, 
conducting research, and solving real-world problems. Interested in Large Language Models, Deep Learning, and 
Agentic AI, with hands-on experience developing transformer-based models, RAG pipelines, and intelligent 
applications. 
EDUCATION 
B. Tech in Electronics & Computer Science Engineering, CGPA: 7.87 
Amrita School of Engineering, Bangalore 2023 – 
2027 
SKILLS 
Programming Languages: Python, C 
AI/ML Frameworks: PyTorch, TensorFlow, Transformers, Scikit-learn, OpenCV, Ultralytics 
LLM Stack: LangChain, LangGraph, LlamaIndex, FAISS, ChromaDB, Hugging Face 
Tools & Data: Streamlit, G

In [ ]:
# ============================================================
# 10. ASK USER QUESTION
# ============================================================

question = input("\nEnter your question: ")


# ============================================================
# 11. GRAPH RETRIEVAL
# ============================================================

retrieved_documents = graph_retriever.invoke(question)


# ============================================================
# 12. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print("\n\nRETRIEVED DOCUMENTS")
print("=" * 60)


for index, document in enumerate(retrieved_documents):

    print(f"\nDOCUMENT {index + 1}")

    print("\nCONTENT:")

    print(document.page_content)

    print("\nMETADATA:")

    print(
        json.dumps(
            document.metadata,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("-" * 60)


# ============================================================
# 13. CREATE CONTEXT
# ============================================================

context = "\n\n".join(
    document.page_content
    for document in retrieved_documents
)


# ============================================================
# 14. SEND RETRIEVED CONTEXT TO GEMINI
# ============================================================

answer_prompt = f"""
Answer the user's question using only the provided context.

Carefully examine ALL retrieved documents before answering.

Identify every project relevant to the question, including projects
whose descriptions or technologies imply relevance even if the exact
term from the question is not present in the project title.

For each relevant project:
1. Give the project name.
2. Explain why it is relevant.
3. List the technologies used.

Do not omit relevant projects.

CONTEXT:

{context}

QUESTION:

{question}
"""

response = llm.invoke(answer_prompt)


# ============================================================
# 15. PRINT FINAL ANSWER
# ============================================================

print("\n\nFINAL ANSWER")
print("=" * 60)

print(response.content)


Enter your question: What evidence shows that Jai Akash has experience with both transformer models and real-world AI applications?


RETRIEVED DOCUMENTS

DOCUMENT 1

CONTENT:
JAI AKASH 
github.com/peteparker123 | huggingface.co/peteparker456 | linkedin.com/in/jai-akash-a3b7a0272 
PROFILE 
Fourth-year Electronics & Computer Science Engineering student passionate about building innovative AI systems, 
conducting research, and solving real-world problems. Interested in Large Language Models, Deep Learning, and 
Agentic AI, with hands-on experience developing transformer-based models, RAG pipelines, and intelligent 
applications. 
EDUCATION 
B. Tech in Electronics & Computer Science Engineering, CGPA: 7.87 
Amrita School of Engineering, Bangalore 2023 – 
2027 
SKILLS 
Programming Languages: Python, C 
AI/ML Frameworks: PyTorch, TensorFlow, Transformers, Scikit-learn, OpenCV, Ultralytics 
LLM Stack: LangChain, LangGraph, LlamaIndex, FAISS, ChromaDB, Hugging Face 
Tools & Data: Streamlit, 